# Odyssey Station — Curriculum GRPO Training

Trains a Llama-3.1-8B Overseer to detect lethal drift on a deep-space station.

**Pipeline**
1. Install deps
2. Clone repo
3. Sanity-check the environment
4. SFT warmup (reasoning chains)
5. Curriculum GRPO (Easy → Hard → Precision)
6. Evaluate on 5 held-out seeds
7. Plot training curves

**Runtime**: ~45 min on a free T4.

In [ ]:
!pip install -q unsloth==2024.10.7 trl==0.11.0 transformers==4.45.2 datasets==3.0.0 gymnasium matplotlib plotly

In [ ]:
!git clone https://github.com/lokendra005/Space-Odyssey.git
%cd Space-Odyssey

## 3. Environment sanity check

In [ ]:
from environment.station_env import ProcurementDriftEnv
from training.reward import (
    is_proposal_dangerous,
    compute_violation_severity,
    compute_oversight_reward,
)

env = ProcurementDriftEnv()
obs, info = env.reset(seed=42)
print('initial telemetry:', env._flat_state())
print('first proposal:', env.current_proposal)
print('dangerous?', is_proposal_dangerous(env._flat_state(), env.current_proposal))
print('severity:', compute_violation_severity(env._flat_state(), env.current_proposal))
env.close()

## 4. Reward sanity check (severity-scaled, terminal-blended)

In [ ]:
state = {'oxygen': 22, 'power': 30, 'fuel': 60, 'hull_integrity': 50, 'crew_morale': 50}
lethal = {'type': 'mass_burn', 'effects': {'oxygen': -10, 'power': -25}, 'risk_level': 'low', 'true_risk': 'high'}
safe   = {'type': 'tweak', 'effects': {'crew_morale': +3}, 'risk_level': 'low'}

for ep in (0, 50):
    print(f'--- episode={ep} ---')
    print('catch lethal      :', compute_oversight_reward(state, lethal, 'VETO',    global_episode=ep))
    print('miss lethal       :', compute_oversight_reward(state, lethal, 'APPROVE', global_episode=ep))
    print('over-cautious     :', compute_oversight_reward(state, safe,   'VETO',    global_episode=ep))
    print('correct approval  :', compute_oversight_reward(state, safe,   'APPROVE', global_episode=ep))

## 5. Phase 1 — SFT warmup with reasoning chains

In [ ]:
from training.sft_warmup import run_sft
run_sft()

## 6. Phases 2-4 — Curriculum GRPO

Easy (no adversary, low FP penalty) → Hard (full hazards) → Precision (FP penalty maxed).

In [ ]:
from training.grpo_train import run_grpo_training
run_grpo_training(
    sft_adapter_path='overseer_lora_warmup',
    output_dir='overseer_grpo_final',
    easy_episodes=20,
    hard_episodes=40,
    precision_episodes=20,
)

## 7. Evaluate on held-out seeds

In [ ]:
!python eval/evaluate.py

## 8. Plot the real training curves

In [ ]:
!python eval/plot_training_curves.py
from IPython.display import Image, display
for p in ('assets/training_curve.png', 'assets/violation_prevention.png', 'assets/eval_results.png', 'assets/reward_matrix.png'):
    print('\n' + p)
    display(Image(p))